# Downstream distribution recovery

After debiasing at layer LL, the network continues processing the modified representation.
This notebook measures whether the downstream layers L > LL "repair" the distribution
back towards what it would have been without any debiasing.

**Definition of recovery:**
  MMD²(raw@L, debiased_from_LL@L) — smaller = more recovered.
  A value of 0 means the distribution at layer L is indistinguishable from raw.

**Two debiasing regimes:**
- **Single** (debias at one layer LL, propagate):  
  Compare `raw/layer_L` vs `debiased/single/{concept}/{method}/from_layer_LL/layer_L`
  for all L >= LL.
- **Sequential** (debias at every layer):  
  Compare `raw/layer_L` vs `debiased/sequential/{concept}/{method}/layer_L_post`
  for all L. Here MMD grows with depth (cumulative effect), not decays.

Analyzed per concept × pos/neg × method.

Data: **test split**, subsampled to max 300 samples/class for MMD.

Output: `notebooks/results/distributions_analysis/downstream_recovery/`

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPTS = [
    'eyeglasses', 'bald', 'chubby', 'wearing_hat', 'male', 'smiling',
    'bags_under_eyes', 'blond_hair', 'double_chin', 'heavy_makeup',
]
METHODS       = ['diff_means', 'lr', 'pclarc']
NUM_LAYERS    = 24
SPLIT         = 'test'
MMD_SUBSAMPLE = 300
MMD_SEED      = 42

# For single debiasing: which debias layers to analyse.
# 'all' = every layer 0..23. Or specify a list, e.g. [0, 6, 12, 18, 23].
SINGLE_DEBIAS_LAYERS = 'all'

RAW_DIR  = ROOT / 'data' / 'activations' / 'raw' / SPLIT
DEB_DIR  = ROOT / 'data' / 'activations' / 'debiased'
METADATA = ROOT / 'data' / 'metadata.csv'
OUT_DIR  = ROOT / 'notebooks' / 'results' / 'distributions_analysis' / 'downstream_recovery'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(),  f'Missing: {RAW_DIR}'
assert METADATA.exists(), f'Missing: {METADATA}'

df_meta  = pd.read_csv(METADATA)
df_split = df_meta[df_meta['split'] == SPLIT].reset_index(drop=True)

single_debias_layers = (
    list(range(NUM_LAYERS))
    if SINGLE_DEBIAS_LAYERS == 'all'
    else SINGLE_DEBIAS_LAYERS
)

print(f'Split            : {SPLIT} ({len(df_split)} images)')
print(f'Concepts         : {len(CONCEPTS)}')
print(f'Methods          : {METHODS}')
print(f'Single deb layers: {single_debias_layers}')

In [ ]:
rng = np.random.default_rng(MMD_SEED)


def subsample(X, n):
    if len(X) <= n:
        return X
    return X[rng.choice(len(X), n, replace=False)]


def mmd_rbf(X, Y):
    """Unbiased MMD² with RBF kernel (median bandwidth heuristic)."""
    X = subsample(X.astype(np.float64), MMD_SUBSAMPLE)
    Y = subsample(Y.astype(np.float64), MMD_SUBSAMPLE)
    n, m = len(X), len(Y)
    if n < 4 or m < 4:
        return np.nan

    XY = np.vstack([X, Y])
    n_med = min(300, len(XY))
    sub = XY[rng.choice(len(XY), n_med, replace=False)]
    sq = np.sum((sub[:, None] - sub[None, :]) ** 2, axis=-1)
    median_sq = np.median(sq[sq > 0])
    if median_sq < 1e-10:
        return 0.0
    gamma = 1.0 / (2.0 * median_sq)

    def rbf(A, B):
        d2 = (np.sum(A**2, axis=1, keepdims=True)
              + np.sum(B**2, axis=1)
              - 2 * A @ B.T)
        return np.exp(-gamma * np.clip(d2, 0, None))

    Kxx, Kyy, Kxy = rbf(X, X), rbf(Y, Y), rbf(X, Y)
    mmd2 = ((Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
            + (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
            - 2 * Kxy.mean())
    return float(max(0.0, mmd2))


def load_split_xy(path, concept):
    """Load parquet at `path`, join concept label, return (X_pos, X_neg).
    Returns (None, None) if file missing.
    """
    if not path.exists():
        return None, None
    df = pd.read_parquet(path).merge(
        df_split[['filename', concept]], on='filename', how='inner'
    )
    feat_cols = [c for c in df.columns if c not in ('filename', concept)]
    X = df[feat_cols].values.astype(np.float32)
    y = df[concept].values
    return X[y == 1], X[y == 0]


def raw_path(layer_idx):
    return RAW_DIR / f'layer_{layer_idx:02d}.parquet'


def single_path(concept, method, debias_layer, downstream_layer):
    return (DEB_DIR / 'single' / concept / method
            / f'from_layer_{debias_layer:02d}' / SPLIT
            / f'layer_{downstream_layer:02d}.parquet')


def seq_post_path(concept, method, layer_idx):
    return (DEB_DIR / 'sequential' / concept / method
            / SPLIT / f'layer_{layer_idx:02d}_post.parquet')

In [ ]:
# ── Single debiasing recovery ──────────────────────────────────────────────────
# For each (concept, method, debias_layer LL), measure MMD(raw@L, debiased@L)
# at every downstream layer L >= LL.

records_single = []

for concept in tqdm(CONCEPTS, desc='Concepts (single)'):
    if concept not in df_split.columns:
        continue

    # Cache raw activations per layer to avoid re-reading
    raw_cache = {}

    for method in METHODS:
        for debias_layer in single_debias_layers:
            for downstream_layer in range(debias_layer, NUM_LAYERS):

                # Load raw at downstream_layer (cache)
                if downstream_layer not in raw_cache:
                    raw_cache[downstream_layer] = load_split_xy(
                        raw_path(downstream_layer), concept
                    )
                X_raw_pos, X_raw_neg = raw_cache[downstream_layer]
                if X_raw_pos is None:
                    continue

                # Load debiased at downstream_layer
                X_deb_pos, X_deb_neg = load_split_xy(
                    single_path(concept, method, debias_layer, downstream_layer),
                    concept
                )
                if X_deb_pos is None:
                    continue

                for label, X_raw, X_deb in [
                    ('pos', X_raw_pos, X_deb_pos),
                    ('neg', X_raw_neg, X_deb_neg),
                ]:
                    records_single.append({
                        'concept':         concept,
                        'method':          method,
                        'debias_layer':    debias_layer,
                        'downstream_layer':downstream_layer,
                        'layers_after':    downstream_layer - debias_layer,
                        'label':           label,
                        'mmd2':            mmd_rbf(X_raw, X_deb),
                    })

df_single = pd.DataFrame(records_single)
df_single.to_csv(OUT_DIR / 'results_single.csv', index=False)
print(f'Single recovery: {len(df_single)} rows saved')

In [ ]:
# ── Sequential debiasing: cumulative drift from raw ────────────────────────────
# At each layer L: MMD(raw@L, sequential_post@L).
# Shows how the distribution cumulatively diverges (or converges) from raw.

records_seq = []

for concept in tqdm(CONCEPTS, desc='Concepts (sequential)'):
    if concept not in df_split.columns:
        continue

    for method in METHODS:
        for layer_idx in range(NUM_LAYERS):
            X_raw_pos, X_raw_neg = load_split_xy(raw_path(layer_idx), concept)
            X_seq_pos, X_seq_neg = load_split_xy(
                seq_post_path(concept, method, layer_idx), concept
            )
            if X_raw_pos is None or X_seq_pos is None:
                continue

            for label, X_raw, X_seq in [
                ('pos', X_raw_pos, X_seq_pos),
                ('neg', X_raw_neg, X_seq_neg),
            ]:
                records_seq.append({
                    'concept': concept,
                    'method':  method,
                    'layer':   layer_idx,
                    'label':   label,
                    'mmd2':    mmd_rbf(X_raw, X_seq),
                })

df_seq = pd.DataFrame(records_seq)
df_seq.to_csv(OUT_DIR / 'results_sequential.csv', index=False)
print(f'Sequential recovery: {len(df_seq)} rows saved')

## Visualization

In [ ]:
# Recovery curves: MMD² vs layers_after_debiasing
# Averaged over concepts and pos/neg; one line per method.
# Separate plot per debias_layer (or a sample of them).

if not df_single.empty:
    plot_layers = sorted(df_single['debias_layer'].unique())
    # Show every 4th layer to keep it readable
    plot_layers = [l for l in plot_layers if l % 4 == 0 or l == plot_layers[-1]]

    n_cols = min(4, len(plot_layers))
    n_rows = int(np.ceil(len(plot_layers) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4 * n_cols, 3.5 * n_rows),
                             sharey=False)
    axes = np.array(axes).flatten()

    for ax, debias_layer in zip(axes, plot_layers):
        sub = df_single[df_single['debias_layer'] == debias_layer]
        for method in METHODS:
            s = (sub[sub['method'] == method]
                 .groupby('layers_after')['mmd2'].mean())
            if s.empty:
                continue
            ax.plot(s.index, s.values, marker='o', ms=3, label=method)
        ax.set_title(f'Debias @ layer {debias_layer}', fontsize=9)
        ax.set_xlabel('Layers after debiasing', fontsize=8)
        ax.set_ylabel('MMD²', fontsize=8)
        if debias_layer == plot_layers[0]:
            ax.legend(fontsize=7)

    for ax in axes[len(plot_layers):]:
        ax.set_visible(False)

    fig.suptitle('Single debiasing — MMD² recovery (raw vs debiased downstream)',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'recovery_curves_single.png', dpi=150)
    plt.show()

In [ ]:
# Heatmap: debias_layer × downstream_layer → mean MMD²
# One heatmap per method, averaged over concepts and pos/neg.

if not df_single.empty:
    for method in METHODS:
        sub = df_single[df_single['method'] == method]
        if sub.empty:
            continue
        pivot = (
            sub.groupby(['debias_layer', 'downstream_layer'])['mmd2']
            .mean()
            .unstack('downstream_layer')
        )
        # Mask upper triangle (downstream < debias doesn't exist)
        mask = pd.DataFrame(
            np.tril(np.ones(pivot.shape, dtype=bool), k=-1),
            index=pivot.index, columns=pivot.columns
        )

        fig, ax = plt.subplots(figsize=(12, 9))
        sns.heatmap(
            pivot, ax=ax, mask=mask,
            cmap='YlOrRd', annot=False, linewidths=0.2,
            cbar_kws={'label': 'MMD²'},
        )
        ax.set_title(
            f'Recovery heatmap — single, method={method}\n'
            f'(row=debias layer, col=downstream layer; '
            f'color=MMD² vs raw, averaged over concepts)',
            fontsize=10
        )
        ax.set_xlabel('Downstream layer')
        ax.set_ylabel('Debiasing layer')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'heatmap_recovery_single_{method}.png', dpi=150)
        plt.show()

In [ ]:
# Sequential: cumulative MMD² from raw across layers

if not df_seq.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, label_filter, title_suffix in [
        (axes[0], None,    'averaged over pos & neg'),
        (axes[1], 'pos',   'positive examples only'),
    ]:
        sub = df_seq if label_filter is None else df_seq[df_seq['label'] == label_filter]
        for method in METHODS:
            s = sub[sub['method'] == method].groupby('layer')['mmd2'].mean()
            if s.empty:
                continue
            ax.plot(s.index, s.values, marker='o', ms=4, label=method)
        ax.set_xlabel('Layer')
        ax.set_ylabel('MMD² (raw vs sequential post)')
        ax.set_title(f'Sequential debiasing drift from raw\n({title_suffix})')
        ax.legend()

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'drift_sequential.png', dpi=150)
    plt.show()

In [ ]:
# Per-concept recovery: does recovery speed differ between concepts?
# Single debiasing, averaged over debias_layers and methods; pos and neg separately.

if not df_single.empty:
    fig, axes = plt.subplots(2, 5, figsize=(18, 8), sharey=False)

    for ax, concept in zip(axes.flatten(), CONCEPTS):
        sub = df_single[df_single['concept'] == concept]
        for label, ls in [('pos', '-'), ('neg', '--')]:
            s = (sub[sub['label'] == label]
                 .groupby('layers_after')['mmd2'].mean())
            if s.empty:
                continue
            ax.plot(s.index, s.values, ls=ls, lw=1.5, label=label)
        ax.set_title(concept, fontsize=9)
        ax.set_xlabel('Layers after deb.', fontsize=8)
        ax.set_ylabel('MMD²', fontsize=8)

    axes[0, 0].legend(fontsize=8)
    fig.suptitle('Recovery speed per concept — single debiasing (avg over methods & debias_layers)',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'recovery_per_concept.png', dpi=150)
    plt.show()

In [ ]:
# Pos vs neg recovery: are positives and negatives affected differently by debiasing?

if not df_single.empty:
    fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 4), sharey=False)
    if len(METHODS) == 1:
        axes = [axes]

    for ax, method in zip(axes, METHODS):
        sub = df_single[df_single['method'] == method]
        for label, ls in [('pos', '-'), ('neg', '--')]:
            s = (sub[sub['label'] == label]
                 .groupby('layers_after')['mmd2'].mean())
            ax.plot(s.index, s.values, ls=ls, lw=2, label=label)
        ax.set_title(method)
        ax.set_xlabel('Layers after debiasing')
        ax.set_ylabel('MMD²')
        ax.legend()

    fig.suptitle('Pos vs neg recovery — single debiasing (avg over concepts & debias_layers)',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'pos_vs_neg_recovery.png', dpi=150)
    plt.show()